# Lab 9- Deep Learning Model

This lab is meant to get you started in using Keras to design Deep Neural Networks. The goal here is to simply repeat your previous lab, but with DNNs.

Let's start with reading the data, like before:

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

filename = "SUSY-small.csv"
VarNames=["signal", "l_1_pT", "l_1_eta","l_1_phi", "l_2_pT", "l_2_eta", "l_2_phi", "MET", "MET_phi", "MET_rel", "axial_MET", "M_R", "M_TR_2", "R", "MT2", "S_R", "M_Delta_R", "dPhi_r_b", "cos_theta_r1"]
RawNames=["l_1_pT", "l_1_eta","l_1_phi", "l_2_pT", "l_2_eta", "l_2_phi","MET", "MET_phi", "MET_rel", "axial_MET"]
FeatureNames=["M_R", "M_TR_2", "R", "MT2", "S_R", "M_Delta_R", "dPhi_r_b", "cos_theta_r1"]

df = pd.read_csv(filename, dtype='float64', names=VarNames)

Now lets define training and test samples. Note that DNNs take very long to train, so for testing purposes we will use only about 10% of the 5 million events in the training/validation sample. Once you get everything working, make the final version of your plots with the full sample. 

Also note that Keras had trouble with the Pandas tensors, so after doing all of the nice manipulation that Pandas enables, we convert the Tensor to a regular numpy tensor.

In [2]:
N_Max=550000
N_Train=500000

Train_Sample=df[:N_Train]
Test_Sample=df[N_Train:N_Max]

X_Train=np.array(Train_Sample[VarNames[1:]])
y_Train=np.array(Train_Sample["signal"])

X_Test=np.array(Test_Sample[VarNames[1:]])
y_Test=np.array(Test_Sample["signal"])


## Exercise 1

You will need to create several models and make sure they are properly trained. Write a function that takes this history and plots the values versus epoch. For every model that you train in the remainder of this lab, assess:

* Has you model's performance plateaued? If not train for more epochs. 
* Compare the performance on training versus test sample. Are you over training?

In [3]:
import matplotlib.pyplot as plt

def plot_history(history, title="Model Performance"):
    plt.figure(figsize=(8,5))

    if "accuracy" in history.history:
        plt.plot(history.history["accuracy"], label="train acc")
        plt.plot(history.history["val_accuracy"], label="val acc")

    plt.plot(history.history["loss"], label="train loss")
    plt.plot(history.history["val_loss"], label="val loss")

    plt.title(title)
    plt.xlabel("Epoch")
    plt.legend()
    plt.show()

## Exercise 2

Following the original paper (see lab 7), make a comparison of the performance (using ROC curves and AUC) between models trained with raw, features, and raw+features data.

In [4]:
X_raw = df[RawNames]
X_feat = df[FeatureNames]
X_all = df[VarNames[1:]]

y = df["signal"]

In [5]:
from sklearn.model_selection import train_test_split

X_train_raw, X_test_raw, y_train, y_test = train_test_split(X_raw, y, test_size=0.2)
X_train_feat, X_test_feat, _, _ = train_test_split(X_feat, y, test_size=0.2)
X_train_all, X_test_all, _, _ = train_test_split(X_all, y, test_size=0.2)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

model_raw = RandomForestClassifier(n_estimators=50)
model_feat = RandomForestClassifier(n_estimators=50)
model_all = RandomForestClassifier(n_estimators=50)

model_raw.fit(X_train_raw, y_train)
model_feat.fit(X_train_feat, y_train)
model_all.fit(X_train_all, y_train)

In [ ]:
plt.figure()

plot_roc(model_raw, X_test_raw, y_test, "Raw")
plot_roc(model_feat, X_test_feat, y_test, "Features")
plot_roc(model_all, X_test_all, y_test, "Raw+Feat")

plt.legend()
plt.show()

## Exercise 3

Design and implement at least 3 different DNN models. Train them and compare performance. You may try different architectures, loss functions, and optimizers to see if there is an effect.

In [ ]:
pip install tensorflow --break-system-packages

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [ ]:
filename = "SUSY.csv"

VarNames = ["signal", "l_1_pT", "l_1_eta","l_1_phi", "l_2_pT", "l_2_eta", 
            "l_2_phi", "MET", "MET_phi", "MET_rel", "axial_MET",
            "M_R", "M_TR_2", "R", "MT2", "S_R", "M_Delta_R", "dPhi_r_b", "cos_theta_r1"]

df = pd.read_csv(filename, names=VarNames)

X = df[VarNames[1:]]
y = df["signal"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
def model_1():
    model = Sequential([
        layers.Dense(32, activation='relu', input_shape=(X_train.shape[1],)),
        layers.Dense(16, activation='relu'),
        layers.Dense(1, activation='sigmoid')
    ])

    model.compile(optimizer='adam',
                  loss='binary_crossentropy',
                  metrics=['accuracy'])
    return model

In [ ]:
def model_2():
    model = Sequential([
        layers.Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
        layers.Dense(64, activation='relu'),
        layers.Dense(32, activation='relu'),
        layers.Dense(1, activation='sigmoid')
    ])

    model.compile(optimizer='adam',
                  loss='binary_crossentropy',
                  metrics=['accuracy'])
    return model

In [ ]:
def model_3():
    model = Sequential([
        layers.Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
        layers.Dropout(0.3),
        layers.Dense(32, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(1, activation='sigmoid')
    ])

    model.compile(optimizer='adam',
                  loss='binary_crossentropy',
                  metrics=['accuracy'])
    return model

In [ ]:
m1 = model_1()
h1 = m1.fit(X_train, y_train,
            validation_data=(X_test, y_test),
            epochs=10,
            batch_size=256)

m2 = model_2()
h2 = m2.fit(X_train, y_train,
            validation_data=(X_test, y_test),
            epochs=10,
            batch_size=256)

m3 = model_3()
h3 = m3.fit(X_train, y_train,
            validation_data=(X_test, y_test),
            epochs=10,
            batch_size=256)

In [ ]:
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt

def plot_roc(model, X_test, y_test, label):
    y_score = model.predict(X_test).ravel()
    fpr, tpr, _ = roc_curve(y_test, y_score)
    roc_auc = auc(fpr, tpr)

    plt.plot(fpr, tpr, label=f"{label} AUC={roc_auc:.3f}")

In [ ]:
plt.figure()

plot_roc(m1, X_test, y_test, "DNN 1")
plot_roc(m2, X_test, y_test, "DNN 2")
plot_roc(m3, X_test, y_test, "DNN 3")

plt.xlabel("FPR")
plt.ylabel("TPR")
plt.legend()
plt.show()

## Exercise 4

Repeat exercise 4 from Lab 8, adding your best performing DNN as one of the models.  


In [ ]:
m1, m2, m3
best_dnn = m3
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt

In [ ]:
def compare_model(model, name, X_train, y_train, X_test, y_test, features=None):
    
    if features is not None:
        Xtr = X_train[features]
        Xte = X_test[features]
    else:
        Xtr = X_train
        Xte = X_test

    if hasattr(model, "fit"):
        model.fit(Xtr, y_train)

    # scoring
    if hasattr(model, "decision_function"):
        scores = model.decision_function(Xte)
    elif hasattr(model, "predict_proba"):
        scores = model.predict_proba(Xte)[:, 1]
    else:
        scores = model.predict(Xte).ravel()

    fpr, tpr, _ = roc_curve(y_test, scores)
    roc_auc = auc(fpr, tpr)

    plt.plot(fpr, tpr, label=f"{name} (AUC={roc_auc:.3f})")

In [ ]:
plt.figure()

compare_model(LogisticRegression(max_iter=1000),
              "Logistic Regression",
              X_train, y_train, X_test, y_test)

compare_model(DecisionTreeClassifier(max_depth=10),
              "Decision Tree",
              X_train, y_train, X_test, y_test)

compare_model(RandomForestClassifier(n_estimators=50),
              "Random Forest",
              X_train, y_train, X_test, y_test)

compare_model(best_dnn,
              "Best DNN",
              X_train, y_train, X_test, y_test)

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Model Comparison (ROC Curves)")
plt.legend()
plt.show()